# Group21 — 오늘 할 일 우선순위 추천 시스템

**문제:** 여러 할 일의 긴급도·중요도·남은 작업량을 비교하여 오늘 먼저 할 일 최대 3개와 추천 이유를 제시합니다.

**실행 방법:** 런타임 → 세션 다시 시작 및 모두 실행. 기본값은 더미 데이터 시연 모드입니다. 실제 할 일을 직접 입력하려면 첫 코드 셀의 `USE_DUMMY_DATA`를 `False`로 수정한 후 다시 실행하세요.

**팀원 및 담당 (계약서 기준)**
- 팀원 A: C1 `collect_tasks()`, 통합 `main()` / 리뷰 담당 팀원 B
- 팀원 B: C2 `add_urgency_scores(tasks)`, C4 `calculate_priority_scores(tasks)` / 리뷰 담당 팀원 C
- 팀원 C: C3 `add_remaining_work(tasks)`, C5 `recommend_top3(tasks)` / 리뷰 담당 팀원 A

**데이터:** 팀 프로젝트 검증용으로 구성한 더미(가상) 데이터입니다. 외부 파일·외부 데이터 출처가 필요하지 않습니다.

**입력:** 할 일 이름, 마감까지 남은 일수(오늘 0, 마감 경과 음수), 중요도 상·중·하, 총 예상 소요 시간(양의 정수, 시간 단위), 완료율(0~100 정수).

**출력:** 추천 순위, 할 일 이름, 총점(최대 15점), 항목별 점수와 추천 이유.

**규칙:** 완료된 일은 제외합니다. 총점 내림차순, 동점이면 남은 일수 오름차순, 중요도 점수 내림차순, 모두 같으면 입력 순서를 유지합니다. 0~2개인 경우도 정상 처리합니다.

**제약:** 단일 사용자, 일 단위 마감, 각 작업 독립 가정. 일정 자동 배분이나 정확한 마감 시각 계산은 이번 범위에서 제외합니다.

**협업 안내:** 아래 코드는 AI 보조 초안입니다. 담당·리뷰 주석은 배정표이며 실제 작업 완료 기록이 아닙니다. 각 팀원이 자신의 핵심 함수를 직접 작성·검증하고 다른 팀원의 함수를 리뷰한 뒤 실제 결과를 남깁니다.



## 0. 더미 데이터 및 실행 모드


In [ ]:
# 담당: 팀원 A / 리뷰: 팀원 B
# AI 보조: 기획서 스키마에 맞춘 더미 데이터 및 시연 모드 구성
# True: 더미 데이터로 실행 / False: 직접 입력
USE_DUMMY_DATA = True

# 더미(가상) 데이터: 동점과 완료된 작업을 포함한 시연용 목록입니다.
DUMMY_TASKS = [
    {"name": "과제 제출", "days_left": 0, "importance": "상", "estimated_hours": 6, "completion_rate": 50},
    {"name": "발표 준비", "days_left": 1, "importance": "상", "estimated_hours": 3, "completion_rate": 0},
    {"name": "보고서 작성", "days_left": 0, "importance": "상", "estimated_hours": 2, "completion_rate": 0},
    {"name": "강의 복습", "days_left": 2, "importance": "중", "estimated_hours": 2, "completion_rate": 0},
    {"name": "자료 정리", "days_left": 8, "importance": "하", "estimated_hours": 1, "completion_rate": 50},
    {"name": "완료된 과제", "days_left": 0, "importance": "상", "estimated_hours": 4, "completion_rate": 100},
]


## C1. 할 일 정보 정리
사용자 입력을 검사하고 완료율 100%인 할 일은 제외합니다. 출력은 표준화된 할 일 목록입니다.


In [ ]:
# 담당: 팀원 A / 리뷰: 팀원 B
# AI 보조: 입력값 검증, 더미/직접 입력 분기 및 완료 작업 제외 초안 작성
def collect_tasks():
    tasks = []

    # 더미 모드에서는 원본을 복사하고 완료된 작업을 제외합니다.
    if USE_DUMMY_DATA:
        for task in DUMMY_TASKS:
            if task["completion_rate"] < 100:
                tasks.append(task.copy())
        return tasks

    # 정수 입력 및 범위 검사를 공통으로 사용합니다.
    def read_int(prompt, minimum=None, maximum=None):
        while True:
            try:
                value = int(input(prompt))
            except ValueError:
                print("정수를 입력해 주세요.")
                continue

            if minimum is not None and value < minimum:
                print(f"{minimum} 이상 입력해 주세요.")
            elif maximum is not None and value > maximum:
                print(f"{maximum} 이하로 입력해 주세요.")
            else:
                return value

    count = read_int("할 일 개수: ", minimum=0)

    for i in range(count):
        print(f"\n[{i + 1}번째 할 일]")

        while True:
            name = input("할 일 이름: ").strip()
            if name != "":
                break
            print("할 일 이름을 입력해 주세요.")

        # 마감이 지난 일도 추천 대상이므로 음수를 허용합니다.
        days_left = read_int("마감까지 남은 일수(오늘 0): ")

        while True:
            importance = input("중요도(상/중/하): ").strip()
            if importance in ["상", "중", "하"]:
                break
            print("상, 중, 하 중 하나를 입력해 주세요.")

        estimated_hours = read_int("총 예상 소요 시간(정수, 시간): ", minimum=1)
        completion_rate = read_int("완료율(0~100%): ", minimum=0, maximum=100)

        if completion_rate == 100:
            print("완료된 일이므로 추천 목록에서 제외합니다.")
            continue

        tasks.append({
            "name": name,
            "days_left": days_left,
            "importance": importance,
            "estimated_hours": estimated_hours,
            "completion_rate": completion_rate,
        })

    return tasks


## C2. 긴급도 점수 계산
C1의 출력 목록을 받아 마감까지 남은 일수에 따라 긴급도 점수(1~5)를 추가합니다.


In [ ]:
# 담당: 팀원 B / 리뷰: 팀원 C
# AI 보조: 기획서의 남은 일수 구간을 조건문으로 옮긴 초안 작성
def add_urgency_scores(tasks):
    for task in tasks:
        days = task["days_left"]
        if days <= 0:
            score = 5
        elif days == 1:
            score = 4
        elif days <= 3:
            score = 3
        elif days <= 7:
            score = 2
        else:
            score = 1
        task["urgency_score"] = score
    return tasks


## C3. 남은 작업량 계산
C2의 출력 목록에 `예상 소요 시간 × (1 - 완료율/100)`으로 구한 남은 작업량(시간)을 추가합니다.


In [ ]:
# 담당: 팀원 C / 리뷰: 팀원 A
# AI 보조: 미완료 비율 계산과 데이터 필드 연결 초안 작성
def add_remaining_work(tasks):
    for task in tasks:
        hours = task["estimated_hours"]
        rate = task["completion_rate"]
        # 같은 수식을 곱셈 후 나눗셈으로 계산하여 경계값 오차를 줄입니다.
        task["remaining_hours"] = hours * (100 - rate) / 100
    return tasks


## C4. 우선순위 점수 계산
C3의 출력 목록에 중요도 점수·작업량 점수·총점을 추가합니다. 총점 = 긴급도 + 중요도 + 작업량(최대 15점).


In [ ]:
# 담당: 팀원 B / 리뷰: 팀원 C
# AI 보조: 중요도 및 작업량 구간별 점수 계산·합산 초안 작성
def calculate_priority_scores(tasks):
    importance_scores = {"상": 5, "중": 3, "하": 1}

    for task in tasks:
        task["importance_score"] = importance_scores[task["importance"]]
        hours = task["remaining_hours"]

        if hours >= 4:
            workload_score = 5
        elif hours >= 3:
            workload_score = 4
        elif hours >= 2:
            workload_score = 3
        elif hours >= 1:
            workload_score = 2
        else:
            workload_score = 1

        task["workload_score"] = workload_score
        task["priority_score"] = (
            task["urgency_score"] + task["importance_score"] + workload_score
        )

    return tasks


## C5. 상위 최대 3개 추천
C4의 총점 목록을 정렬하고 동점 처리 규칙에 따라 최대 3개를 선택해 추천 이유를 추가합니다.


In [ ]:
# 담당: 팀원 C / 리뷰: 팀원 A
# AI 보조: 동점 정렬 기준, 최대 3개 선택, 추천 이유 생성 초안 작성
def recommend_top3(tasks):
    # 총점은 높은 순, 남은 일수는 적은 순, 중요도는 높은 순입니다.
    # sorted는 모든 기준이 같으면 기존 목록 순서를 유지합니다.
    sorted_tasks = sorted(tasks, key=lambda task: (
        -task["priority_score"],
        task["days_left"],
        -task["importance_score"],
    ))

    result = []
    for i, task in enumerate(sorted_tasks[:3], start=1):
        recommended = task.copy()
        recommended["rank"] = i
        days = recommended["days_left"]
        if days < 0:
            deadline = f"마감 {-days}일 경과"
        elif days == 0:
            deadline = "오늘 마감"
        else:
            deadline = f"마감 {days}일 남음"

        recommended["reason"] = (
            f"{deadline}(긴급도 {task['urgency_score']}점), "
            f"중요도 {task['importance']}({task['importance_score']}점), "
            f"남은 작업량 {task['remaining_hours']:.2f}시간({task['workload_score']}점)"
        )
        result.append(recommended)

    return result


## 통합. main — Output → Input
C1의 목록을 C2, C3, C4, C5에 차례로 전달합니다. 추천할 일이 없으면 안내하고, 있으면 순위·이름·총점·추천 이유를 출력합니다.


In [ ]:
# 담당: 팀원 A / 리뷰: 팀원 B
# AI 보조: 다섯 함수의 반환값 연결 및 결과 출력 초안 작성
def main():
    print("=== Group21 오늘 할 일 우선순위 추천 시스템 ===")
    print("실행 모드:", "더미(가상) 데이터" if USE_DUMMY_DATA else "직접 입력")

    # 앞 함수의 Output을 다음 함수의 Input으로 전달합니다.
    tasks = collect_tasks()                     # C1
    tasks = add_urgency_scores(tasks)            # C2
    tasks = add_remaining_work(tasks)            # C3
    tasks = calculate_priority_scores(tasks)    # C4
    recommendations = recommend_top3(tasks)      # C5

    print(f"추천 대상: 미완료 할 일 {len(tasks)}개")
    if len(recommendations) == 0:
        print("추천할 할 일이 없습니다.")
        return

    print("\n[오늘 우선할 일]")
    for task in recommendations:
        print(f"{task['rank']}위. {task['name']} — {task['priority_score']}/15점")
        print(f"   추천 이유: {task['reason']}")


## 테스트 — 정상·엣지·에러
예상값과 실제 반환값을 비교합니다. `assert`는 조건이 틀리면 테스트 실패로 중단합니다.

`unittest.mock`은 테스트에서만 실제 사용자의 입력을 대신하기 위해 쓰는 Python 표준 도구입니다. 핵심 추천 로직은 외부 라이브러리를 사용하지 않습니다.


In [ ]:
# 담당: 팀원 A / 리뷰: 팀원 B
# AI 보조: 정상·동점·엣지·잘못된 입력 자동 검증 사례 작성
from unittest.mock import patch
from contextlib import redirect_stdout
from io import StringIO


def run_pipeline_for_test(tasks):
    # 테스트 전용 함수: C2부터 C5까지 실행합니다.
    tasks = [task.copy() for task in tasks]
    tasks = add_urgency_scores(tasks)
    tasks = add_remaining_work(tasks)
    tasks = calculate_priority_scores(tasks)
    return recommend_top3(tasks)


def make_test_task(name, days, importance, hours, completion=0):
    # 테스트용 더미 Dict를 만듭니다.
    return {"name": name, "days_left": days, "importance": importance,
            "estimated_hours": hours, "completion_rate": completion}


previous_mode = USE_DUMMY_DATA
try:
    # 정상: 실제 C1부터 실행해 더미 목록의 완료 작업 제외 및 추천 결과를 확인합니다.
    USE_DUMMY_DATA = True
    original = collect_tasks()
    normal_results = run_pipeline_for_test(original)
    assert len(original) == 5
    assert [task["name"] for task in normal_results] == ["과제 제출", "보고서 작성", "발표 준비"]
    assert [task["priority_score"] for task in normal_results] == [14, 13, 13]
    assert normal_results[0]["remaining_hours"] == 3
    assert "완료된 과제" not in [task["name"] for task in original]
    print("정상 PASS: 과제 제출 14점 > 보고서 작성 13점 > 발표 준비 13점 (마감 동점 처리)")
    print("완료 제외 PASS: 완료율 100%인 작업은 입력 목록에서 제외")

    # 동점: 총점과 마감이 같으면 중요도가 우선하고 전부 같으면 입력 순서 유지.
    tie = run_pipeline_for_test([
        make_test_task("중요도 중", 0, "중", 3),
        make_test_task("중요도 상", 0, "상", 1),
    ])
    assert [task["priority_score"] for task in tie] == [12, 12]
    assert tie[0]["name"] == "중요도 상"
    full_tie = run_pipeline_for_test([
        make_test_task("먼저 입력", 2, "중", 2),
        make_test_task("나중 입력", 2, "중", 2),
    ])
    assert [task["name"] for task in full_tie] == ["먼저 입력", "나중 입력"]
    print("동점 PASS: 중요도 우선, 모든 조건 동일 시 입력 순서 유지")

    # 엣지: 미완료 작업이 1~2개이거나 하나도 없을 때 정상 처리.
    assert len(run_pipeline_for_test(original[:1])) == 1
    assert len(run_pipeline_for_test(original[:2])) == 2
    assert run_pipeline_for_test([]) == []
    print("엣지 PASS: 1개·2개 추천 및 0개 빈 결과")

    # 긴급도 구간 경계, 마감이 지난 작업 포함.
    urgency_cases = [make_test_task(str(d), d, "중", 1) for d in [-1, 0, 1, 2, 3, 4, 7, 8]]
    urgency_results = add_urgency_scores(urgency_cases)
    assert [task["urgency_score"] for task in urgency_results] == [5, 5, 4, 3, 3, 2, 2, 1]

    # 작업량 구간 경계와 중요도 3종 검증.
    work_cases = [make_test_task(str(h), 0, "중", h) for h in [1, 2, 3, 4]]
    work_cases.append(make_test_task("1시간 미만", 0, "하", 1, 50))
    work_cases = calculate_priority_scores(add_remaining_work(add_urgency_scores(work_cases)))
    assert [task["workload_score"] for task in work_cases] == [2, 3, 4, 5, 1]
    assert [task["importance_score"] for task in work_cases] == [3, 3, 3, 3, 1]
    print("경계값 PASS: 긴급도와 작업량의 점수 구간 확인")

    # 에러: 실제 C1에 잘못된 입력을 넣고 유효한 값이 올 때까지 재입력하는지 검증.
    USE_DUMMY_DATA = False
    invalid_inputs = ["abc", "-1", "1", "", "테스트 작업", "문자", "-2",
                      "높음", "상", "-3", "2", "101", "20"]
    captured = StringIO()
    with patch("builtins.input", side_effect=invalid_inputs), redirect_stdout(captured):
        validated = collect_tasks()
    assert validated == [make_test_task("테스트 작업", -2, "상", 2, 20)]
    assert "정수를 입력" in captured.getvalue()
    assert "상, 중, 하" in captured.getvalue()
    assert "100 이하" in captured.getvalue()
    print("에러 PASS: 문자·음수 시간·범위 초과·잘못된 중요도·빈 이름 재입력")

    # 실제 main 경로에서 입력 0개와 모두 완료인 경우의 안내 메시지 확인.
    for answers in [["0"], ["1", "완료된 일", "0", "상", "2", "100"]]:
        captured = StringIO()
        with patch("builtins.input", side_effect=answers), redirect_stdout(captured):
            main()
        assert "추천할 할 일이 없습니다." in captured.getvalue()
    print("빈 목록 PASS: main에서 추천 대상 없음 안내")
    print("\n전체 테스트 통과")

finally:
    # 테스트가 실행 모드 설정을 바꾸지 않도록 원래 값을 복원합니다.
    USE_DUMMY_DATA = previous_mode


정상 PASS: 과제 제출 14점 > 보고서 작성 13점 > 발표 준비 13점 (마감 동점 처리)
완료 제외 PASS: 완료율 100%인 작업은 입력 목록에서 제외
동점 PASS: 중요도 우선, 모든 조건 동일 시 입력 순서 유지
엣지 PASS: 1개·2개 추천 및 0개 빈 결과
경계값 PASS: 긴급도와 작업량의 점수 구간 확인
에러 PASS: 문자·음수 시간·범위 초과·잘못된 중요도·빈 이름 재입력
빈 목록 PASS: main에서 추천 대상 없음 안내

전체 테스트 통과


## 실행 결과
전체 프로그램을 처음부터 끝까지 실행합니다. 기본 모드는 더미 데이터이므로 모든 셀을 한 번에 실행할 수 있습니다.


In [ ]:
# 담당: 팀원 A / 리뷰: 팀원 B
# AI 보조: 통합 실행 셀 구성
main()


=== Group21 오늘 할 일 우선순위 추천 시스템 ===
실행 모드: 더미(가상) 데이터
추천 대상: 미완료 할 일 5개

[오늘 우선할 일]
1위. 과제 제출 — 14/15점
   추천 이유: 오늘 마감(긴급도 5점), 중요도 상(5점), 남은 작업량 3.00시간(4점)
2위. 보고서 작성 — 13/15점
   추천 이유: 오늘 마감(긴급도 5점), 중요도 상(5점), 남은 작업량 2.00시간(3점)
3위. 발표 준비 — 13/15점
   추천 이유: 마감 1일 남음(긴급도 4점), 중요도 상(5점), 남은 작업량 3.00시간(4점)
